# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [69]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


In [70]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [71]:
spaceship.dropna(inplace=True) 


In [72]:
spaceship['Cabin'] = spaceship['Cabin'].str.partition('/')[0]


In [73]:
spaceship = spaceship.drop(columns=['PassengerId', 'Name'])


In [74]:
cols = ['CryoSleep', 'VIP']
spaceship[cols] = spaceship[cols].astype(int)
cols = ['HomePlanet', 'Destination', 'Cabin']

for col in cols:
    dummies = pd.get_dummies(
        spaceship[col],
        prefix=col
    )

    spaceship = pd.concat([spaceship, dummies], axis=1)

spaceship = spaceship.drop(columns=['HomePlanet', 'Destination', 'Cabin'])


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [75]:
features = spaceship.drop(columns=["Transported"])
target = spaceship["Transported"]

In [76]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = 0.20, random_state=0)

In [77]:
scaler = StandardScaler()

X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.transform(X_test)

In [78]:
X_train_norm = pd.DataFrame(X_train_norm,columns=X_train.columns)

X_test_norm = pd.DataFrame(X_test_norm,columns=X_test.columns)

- Evaluate your model

In [79]:
bagging_reg = BaggingRegressor(DecisionTreeRegressor(max_depth=20),
                               n_estimators=100,
                               max_samples = 1000)

In [80]:
bagging_reg.fit(X_train_norm, y_train)

BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=20),
                 max_samples=1000, n_estimators=100)

In [81]:
pred = bagging_reg.predict(X_test_norm)

print("MAE", mean_absolute_error(y_test, pred))

rmse = np.sqrt(mean_squared_error(y_test, pred))
print("RMSE", rmse)

print("R2 score", bagging_reg.score(X_test_norm, y_test))

MAE 0.27774031932594573
RMSE 0.37775385346550444
R2 score 0.4292081047678489


**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [ ]:
# Base model
bagging_reg = BaggingRegressor(DecisionTreeRegressor(max_depth=20),
                               n_estimators=100,
                               max_samples = 1000)
# Parameter distributions
param_dist = {
    'n_estimators': [50, 100, 150, 200, 300],
    'max_samples': [500, 800, 1000, 1200, 1500],
    'estimator__max_depth': [5, 10, 15, 20, 25, 30, None],
    'estimator__min_samples_split': [2, 5, 10, 20],
    'estimator__min_samples_leaf': [1, 2, 4, 8]
}

# Random Search
random_search = RandomizedSearchCV(
    estimator=bagging_reg,
    param_distributions=param_dist,
    n_iter=50,
    cv=5,
    scoring={
        'r2': 'r2',
        'mae': 'neg_mean_absolute_error',
        'rmse': 'neg_root_mean_squared_error'
    },
    refit='r2',  # best model selected using R²
    n_jobs=-1,
    random_state=42,
    verbose=2
)

# Fit
random_search.fit(X_train, y_train)

# Best model
best_model = random_search.best_estimator_

y_pred = best_model.predict(X_test)

# Metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Best Parameters:", random_search.best_params_)
print(f"CV R²: {random_search.best_score_:.4f}")
print(f"Test R²: {r2:.4f}")
print(f"Test MAE: {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best Parameters: {'n_estimators': 300, 'max_samples': 1500, 'estimator__min_samples_split': 20, 'estimator__min_samples_leaf': 2, 'estimator__max_depth': None}
CV R²: 0.4555
Test R²: 0.4309
Test MAE: 0.2769
Test RMSE: 0.3772


- Run Grid Search

In [84]:
# Base model
bagging_reg = BaggingRegressor(DecisionTreeRegressor(max_depth=20),
                               n_estimators=100,
                               max_samples = 1000)

# Hyperparameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_samples': [500, 1000, 2000],
    'estimator__max_depth': [10, 20, 30, None],
    'estimator__min_samples_split': [2, 5, 10]
}

# Grid Search
grid_search = GridSearchCV(
    estimator=bagging_reg,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=2
)

# Fit
grid_search.fit(X_train, y_train)

# Best model
best_model_2 = grid_search.best_estimator_

y_pred = best_model_2.predict(X_test)

# Metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Best Parameters:", random_search.best_params_)
print(f"CV R²: {random_search.best_score_:.4f}")
print(f"Test R²: {r2:.4f}")
print(f"Test MAE: {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best Parameters: {'n_estimators': 300, 'max_samples': 1500, 'estimator__min_samples_split': 20, 'estimator__min_samples_leaf': 2, 'estimator__max_depth': None}
CV R²: 0.4555
Test R²: 0.4330
Test MAE: 0.2797
Test RMSE: 0.3765


- Evaluate your model

|Model Bagging Regressor | Grid Search|
|:----:|:-----:|
| **Metric** | **Value** |
|**R2**|0.429|
|**MAE**|0.278|
|**RMSE**|0.377|
|**Model Bagging Regressor**| **Random Search**|
| **Metric** | **Value** |
|**R2**|0.430|
|**MAE**|0.276|
|**RMSE**|0.377|
|**Model Bagging Regressor** | **Grid Search**|
| **Metric** | **Value** |
|**R2**|0.433|
|**MAE**|0.279|
|**RMSE**|0.376|

Grid Search improve slightly the R2 from the model 